# CryptoRisk & Trend SBC
**Sistema Basado en Conocimiento para evaluación de riesgo y decisiones de *swing trading* en criptomonedas**

Proyecto Tercer Corte · IAIA · Integrantes: *COMPLETE*

> Fines académicos. No constituye asesoría financiera.

## 1. Dominio y justificación
*COMPLETE:* describa el dominio (análisis técnico + macro), por qué es adecuado para reglas de Horn
(jerarquía datos → estados → recomendación, incertidumbre, trazabilidad) y su viabilidad.
Cite a Russell & Norvig (2021, cap. 7).

In [ ]:
# Imports del motor (copiado del laboratorio en motor/motor.py)
from motor import *


## 2. Modelado formal

### 2.1 Proposiciones atómicas (11)
| symbol | meaning | nivel |
|---|---|---|
| precio_sobre_ema200 | el precio está por encima de la EMA de 200 periodos | entrada |
| rsi_sobreventa | el RSI está por debajo de 30 | entrada |
| rsi_sobrecompra | el RSI está por encima de 70 | entrada |
| volumen_creciente | el volumen de las últimas sesiones va en aumento | entrada |
| soporte_tecnico_clave | el precio rebotó en un soporte histórico importante | entrada |
| noticias_macro_negativas | hay noticias macro o regulatorias adversas | entrada |
| tendencia_saludable | el mercado mantiene una estructura de precios favorable | intermedio |
| alerta_correccion | hay indicios técnicos de posible retroceso | intermedio |
| oportunidad_compra | las condiciones sugieren un buen punto de entrada | intermedio |
| recomendar_compra | abrir posición de compra (riesgo moderado/bajo) | meta |
| recomendar_caja | mantener liquidez / evitar operar (alto riesgo) | meta |

### 2.2 Reglas (cláusulas de Horn)
- R1: precio_sobre_ema200 ∧ soporte_tecnico_clave ⇒ tendencia_saludable
- R2: rsi_sobrecompra ∧ noticias_macro_negativas ⇒ alerta_correccion
- R3: rsi_sobreventa ∧ volumen_creciente ⇒ oportunidad_compra
- R4: rsi_sobreventa ∧ soporte_tecnico_clave ⇒ oportunidad_compra *(respaldo si el volumen es UNKNOWN)*
- R5: tendencia_saludable ∧ oportunidad_compra ⇒ recomendar_compra
- R6a: alerta_correccion ⇒ recomendar_caja
- R6b: noticias_macro_negativas ⇒ recomendar_caja

*COMPLETE:* explique por qué `A ∨ B ⇒ C` se divide en dos reglas (Horn: un solo literal positivo, sin disyunción en el antecedente).

## 3. Valores YES / NO / UNKNOWN
*COMPLETE:* explique la diferencia entre átomo **ausente** de la FB y átomo con `UNKNOWN` (memoria: no se vuelve a preguntar) y cómo R4 evita el bloqueo cuando `volumen_creciente` es `UNKNOWN`.

## 4. Carga de la KB con el Subsistema de Desarrollo

In [ ]:
# PSEUDOCODIGO:
#   kb = TKnowledgeBase()
#   dev = DevelopmentSubsystem(kb)
#   1) Crear los 11 TAtom (symbol, meaning) y registrarlos con kb.add(...)
#        - puede hacerse por código para la KB completa
#        - y DEMOSTRAR dev.add_atom()/dev.add_rule() con al menos 1 átomo y 1 regla
#   2) Crear R1, R2, R3, R4, R5, R6a, R6b con TRule(nombre, [TAtom...], TAtom) y kb.add(...)
#   3) Demostrar la validación: intentar agregar una regla con un átomo NO registrado
#      -> debe lanzar ValueError (capturarlo e imprimir el mensaje)
#   4) dev.show()  # mostrar la KB construida


## 5. Forward chaining

### Escenario 1 — Mercado alcista
FB inicial (`user`): `precio_sobre_ema200=YES`, `soporte_tecnico_clave=YES`, `rsi_sobreventa=YES`, `volumen_creciente=YES`.

Esperado: R1 → tendencia_saludable; R3 → oportunidad_compra; R5 → **recomendar_compra**.

In [ ]:
# PSEUDOCODIGO:
#   fb1 = TFactBase()
#   fb1.set(<átomo>, TValue.YES, "user")  para los 4 hechos
#   forward(kb, fb1)
#   imprimir la FB resultante (use QuerySubsystem.show() o un bucle)
#   imprimir explain(kb, fb1, recomendar_compra)


### Escenario 2 — Incertidumbre: `volumen_creciente = UNKNOWN`
FB inicial: `precio_sobre_ema200=YES`, `soporte_tecnico_clave=YES`, `rsi_sobreventa=YES`, `volumen_creciente=UNKNOWN (user)`.

Esperado: R3 queda bloqueada; **R4** deduce oportunidad_compra; R1 y R5 → recomendar_compra.

In [ ]:
# PSEUDOCODIGO:
#   fb2 = TFactBase(); cargar los 4 hechos (el volumen con TValue.UNKNOWN)
#   forward(kb, fb2)
#   mostrar FB y explain(...). Verificar que oportunidad_compra tiene origen R4 (no R3)
#   comentario: ¿qué pasaría sin R4?


### Escenario 3 — Mercado bajista
FB inicial: `precio_sobre_ema200=NO`, `rsi_sobrecompra=YES`, `noticias_macro_negativas=YES`.

Esperado: R2 → alerta_correccion; R6a → **recomendar_caja** (R6b también es aplicable).

In [ ]:
# PSEUDOCODIGO:
#   fb3 = TFactBase(); cargar hechos
#   forward(kb, fb3); mostrar FB y explain(kb, fb3, recomendar_caja)
#   comentario: ¿por qué recomendar_compra NO se deduce? (R1 queda en NO por el antecedente NO)


## 6. Backward chaining (interactivo)

### Escenario 4 — Meta `recomendar_compra` con FB vacía
El motor recorre R5 → R1 / R3 / R4 y **pregunta solo por átomos de entrada** (los que ninguna regla concluye).
Orden esperado de preguntas: `precio_sobre_ema200`, `soporte_tecnico_clave`, `rsi_sobreventa`, `volumen_creciente`.

Responda en la ejecución interactiva y pruebe al menos dos recorridos: uno con todo `YES` y otro respondiendo `UNKNOWN` al volumen (para que se use R4).
Nota: `backward` devuelve `UNKNOWN` (no `NO`) cuando ninguna regla logra concluir la meta.

In [ ]:
# PSEUDOCODIGO:
#   fb4 = TFactBase()
#   valor, fb4 = backward(kb, fb4, recomendar_compra)   # responde el usuario con input()
#   imprimir valor y la FB resultante
#   imprimir explain(kb, fb4, recomendar_compra)
#   (para la grabación/presentación: anote el orden de las preguntas)


## 7. Explicabilidad (cómo y por qué)

In [ ]:
# PSEUDOCODIGO:
#   CÓMO: mostrar explain(...) de la conclusión de los escenarios 1, 2 y 3
#         (salida indentada: conclusión <- regla, hasta hechos 'user')
#   POR QUÉ: para una pregunta del backward (p. ej. volumen_creciente),
#         imprimir why(kb, recomendar_compra, volumen_creciente)


## 8. Limitación: conflicto de metas
Con noticias macro negativas **y** condiciones técnicas favorables se pueden deducir `recomendar_compra` y `recomendar_caja` a la vez.

In [ ]:
# PSEUDOCODIGO:
#   fb5 = TFactBase(); los 4 hechos favorables del Escenario 1 + noticias_macro_negativas=YES
#   forward(kb, fb5)
#   comprobar que ambas metas quedan en YES
#   propuesta: función resolver_conflicto(fb) que dé prioridad a recomendar_caja


## 9. Conclusiones
*COMPLETE:* ventajas de la IA simbólica en finanzas (transparencia, trazabilidad, manejo de incertidumbre),
limitaciones (umbrales fijos, sin grados de certeza, conflicto de metas) y trabajo futuro.

## Referencias
- Russell, S. & Norvig, P. (2021). *Artificial Intelligence: A Modern Approach* (4.ª ed.), cap. 7.
- Laboratorio 05 · IAIA 2026-2.